# 01. Azure Machine Learning 環境構築

**対応するテキスト**: [docs/03_AzureML環境構築.md](../docs/03_AzureML環境構築.md)

このノートブックで行うこと:

1. **ローカル環境のセットアップ**（パッケージの導入と検証）
2. ワークスペースへの接続（**Azure へのサインインもこの中で行います**）
3. コンピューティング クラスターの作成（`min_instances=0`）
4. カスタム環境（panda-gym + Stable-Baselines3）の作成
5. **疎通確認ジョブの実行と MLflow 記録の確認**

> **ターミナルは不要です。**
> パッケージの導入も Azure へのサインインも、**このノートブックのセルを実行するだけ**で完了します。

> ⚠ **実行前に必ず [docs/02_Azure環境の準備.md](../docs/02_Azure環境の準備.md) のチェックリストを完了してください。**
> 特に **権限**と **vCPU クォータ**が不足していると、この先すべて失敗します。


## 1. セットアップ

**このセクションのセルを実行するだけで、ローカル環境が完成します。ターミナルを開く必要はありません。**

### 前提

| # | 必要なもの | 理由 |
|---|---|---|
| 1 | VS Code + Python 拡張機能（または JupyterLab） | Notebook を実行するため |
| 2 | **本ハンズオン専用の Python カーネル** | このセクションは**いま選択しているカーネルの環境を直接書き換えます** |
| 3 | （1-3 を実行する場合のみ）**conda 環境であること** | `pybullet` を conda-forge から導入するため（理由は 1-3 に記載） |

> [!IMPORTANT]
> **Python が 1 つも入っていない PC では、この Notebook 自体を開けません。**
> その場合だけは、先に conda（Miniforge）を導入してください。
> 一括導入スクリプトが [../setup/setup.ps1](../setup/setup.ps1)（Windows）と [../setup/setup.sh](../setup/setup.sh)（macOS / Linux）にあります。

> [!WARNING]
> **他の用途と共用している環境では実行しないでください。**
> Azure ML コンピューティング インスタンスの既定環境（`azureml_py310_sdkv2` など）は特に対象外です。
> それらの環境では **1-2 と 1-3 は不要**なことがほとんどです（1-1 の判定で確認できます）。

### 1-1. 環境診断

**次のセルは何も導入しません。** 何が足りないかを調べるだけなので、何度実行しても安全です。


In [ ]:
# ============================================================
#  1-1. 環境診断（何も導入しません）
# ============================================================
import importlib.metadata as metadata
import os
import shutil
import sys
from pathlib import Path

#  1-2 で導入する。2. 以降（ワークスペース接続・ジョブ投入・MLflow 参照）に必須
AZURE_PACKAGES = ("azure-ai-ml", "azure-identity", "mlflow", "azureml-mlflow")

#  1-3 で導入する。ローカルで RL 環境を動かす場合のみ必要
RL_PACKAGES = (
    "pybullet",
    "numpy",
    "scipy",
    "gymnasium",
    "panda-gym",
    "stable-baselines3",
    "imageio",
    "imageio-ffmpeg",
    "pandas",
    "matplotlib",
)


def get_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def report(title, names):
    print(title)
    missing = []
    for name in names:
        version = get_version(name)
        print(f"  {name:<20}{version or '（未導入）'}")
        if version is None:
            missing.append(name)
    return missing


print("[実行中のカーネル]")
print(f"  python     : {sys.version.split()[0]}")
print(f"  executable : {sys.executable}")

# conda 環境の直下には必ず conda-meta フォルダーが存在する
is_conda_env = (Path(sys.prefix) / "conda-meta").is_dir()
conda_exe = shutil.which("conda") or os.environ.get("CONDA_EXE")
print(f"  conda 環境 : {'はい' if is_conda_env else 'いいえ'}")
print(f"  conda      : {conda_exe or '見つかりません'}")
print()

missing_azure = report("[1-2 Azure ML SDK / MLflow — 2. 以降で必須]", AZURE_PACKAGES)
print()
missing_rl = report("[1-3 ローカル RL 実行用 — 1-5 / 1-6 を使う場合のみ]", RL_PACKAGES)

numpy_version = get_version("numpy")
numpy_too_new = numpy_version is not None and int(numpy_version.split(".")[0]) >= 2

print()
print("[判定]")
print(f"  1-2 の実行 : {'必要' if missing_azure else '不要（導入済み）'}")
if numpy_too_new:
    print(f"  1-3 の実行 : 必要（numpy {numpy_version} は panda-gym の numpy<2 制約に反します）")
else:
    print(f"  1-3 の実行 : {'必要' if missing_rl else '不要（導入済み）'}")

if (missing_rl or numpy_too_new) and not is_conda_env:
    print("  ⚠ このカーネルは conda 環境ではないため、1-3 の conda セルは失敗します。1-3 の説明を読んでください。")


### 1-2. Azure ML SDK と MLflow の導入

**このノートブックの 2. 以降（ワークスペース接続・ジョブ投入・MLflow 参照）に必須です。**

`%pip` は、**いま動いているカーネルの環境に対して** pip を実行する IPython のマジック コマンドです。
そのため、ターミナルを開いて `conda activate` する必要がありません。

> **出典（参考情報・OSS 公式ソース）**: [IPython — Built-in magic commands](https://ipython.readthedocs.io/en/stable/interactive/magics.html)
> 「**`%pip`** … Run the pip package manager **within the current kernel**.」

> [!NOTE]
> **1-1 の判定が「不要（導入済み）」なら、このセルは飛ばして構いません。**
> 実行しても害はありません（pip が「already satisfied」と表示して終わります）。


In [ ]:
%pip install azure-ai-ml azure-identity mlflow azureml-mlflow


### 1-3.【任意】ローカルで RL 環境を動かすための導入

> [!NOTE]
> **この節は「手元の PC で panda-gym を動かしたい人」だけが対象です。**
>
> - **Azure ML にジョブを投げて MLflow で結果を見るだけなら不要です。** 1-2 だけで足ります。
> - Azure ML のコンピューティングは **Linux** なので、[../src/conda.yaml](../src/conda.yaml) 側にはこの節は一切関係しません。
> - **ロボットが動く様子を 3D GUI で見る**（[04 章](../docs/04_RL環境を触って理解する.md) 4.6）のは、**ローカル実行でしかできません。** そのために必要なのがこの節です。

#### なぜ `pip install panda-gym` だけでは足りないのか

`panda-gym` は物理エンジン **`pybullet`** に依存します。
この `pybullet` は、**PyPI に Windows 向けと macOS 向けのビルド済みホイールを公開していません。**
最新版 3.2.7 の配布物は **manylinux（Linux）向けホイール・PyPy 向けホイール・ソース配布 (`.tar.gz`) だけ**です。

そのため Windows で `pip install panda-gym` を実行すると **ソースからのビルド**に入り、C++ コンパイラーが無い環境では次のエラーで失敗します。

```
error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools":
https://visualstudio.microsoft.com/visual-cpp-build-tools/
```

一方 **conda-forge は `win-64` 向けのビルド済み `pybullet` を配布しています。**
そこで **`pybullet` だけを conda-forge から入れ、残りを `pip` で入れる**と、**C++ コンパイラーなしで導入できます。**

> **出典（参考情報・サードパーティ）**
> - PyPI の `pybullet` 配布ファイル一覧（Windows / macOS 向けホイールが無いこと）: https://pypi.org/project/pybullet/#files
> - conda-forge の `pybullet` 配布パッケージ一覧（`win-64` を含むこと）: https://anaconda.org/conda-forge/pybullet
> - `panda-gym` の依存定義（`gymnasium>=0.26`, `pybullet`, `numpy<2`, `scipy`）: https://github.com/qgallouedec/panda-gym/blob/master/setup.py
> - Microsoft C++ Build Tools: https://visualstudio.microsoft.com/visual-cpp-build-tools/

#### なぜ `%conda` マジックを使わないのか

IPython には `%pip` と対になる **`%conda`** マジックがあります。しかし**この場面では使えません。**

`%conda` は、渡された行を **`shlex.split()` で分解して引用符を取り除いてから**、シェル経由でコマンドを実行します。
そのため `%conda install ... "numpy<2"` と書いても**引用符が失われ、`<` がシェルのリダイレクト演算子として解釈されてしまいます。**

conda 公式ドキュメントも「シェルに指定を入力するときは、`<`・`>`・`*`・`|` のようにシェルが解釈する文字を含む指定は引用符で囲むこと」と明記しています。
そこで次のセルでは、**conda を引数リストで直接呼び出し、シェルをまったく経由しません。** これなら `numpy<2` がそのまま conda に届きます。

> **出典（参考情報・OSS 公式ソース）**: IPython の `%conda` 実装（`_run_command` が `shlex.split(line)` を行い、結果を `self.shell.system(...)` に渡している）
> https://github.com/ipython/ipython/blob/main/IPython/core/magics/packaging.py

> **出典（参考情報・OSS 公式ソース）**: [conda — Package match specifications](https://docs.conda.io/projects/conda/en/latest/user-guide/concepts/pkg-specs.html)
> 「When entering package specifications in a shell, quote specifications that contain characters interpreted by the shell, such as `<`, `>`, `*`, or `|`.」

なお **`%pip` は行をそのまま渡すため引用符が保たれます。** 2 つ目のセル（pip）は `%pip` をそのまま使います。

> **出典（参考情報・OSS 公式ソース）**: [IPython — Built-in magic commands](https://ipython.readthedocs.io/en/stable/interactive/magics.html)
> 「**`%pip`** … Run the pip package manager **within the current kernel**.」

#### 実行する前に

> [!WARNING]
> **次の 2 つのセルは、いま選択しているカーネルの環境を直接書き換えます。**
>
> - `numpy` が 2.x の場合は **1.x へダウングレードされます**（`panda-gym` の必須制約）。
> - **他の用途と共用している環境では実行しないでください。**
> - 1-1 で **「conda 環境: いいえ」** と表示された場合、次のセルは失敗します。conda 環境を用意する手順は [../setup/setup.ps1](../setup/setup.ps1) / [../setup/setup.sh](../setup/setup.sh) にあります。
> - **完了までに時間がかかります。** `stable-baselines3` が PyTorch を引き込むためで、異常ではありません。


In [ ]:
# ============================================================
#  1-3 (1/2) pybullet・numpy・scipy を conda-forge から導入する
#  ※ conda をシェル経由ではなく引数リストで直接呼び出します（理由は上の説明を参照）
# ============================================================
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not (Path(sys.prefix) / "conda-meta").is_dir():
    raise RuntimeError(
        f"このカーネル（{sys.prefix}）は conda 環境ではないため、conda で導入できません。"
        " 1-3 の説明と 1-7 のつまずきポイントを参照してください。"
    )


def find_conda():
    """現在のカーネルから使える conda 実行ファイルを探す。"""
    #  conda で activate された環境では CONDA_EXE が設定されている
    exe = os.environ.get("CONDA_EXE")
    if exe and Path(exe).is_file():
        return exe
    exe = shutil.which("conda")
    if exe:
        return exe
    #  base 環境なら sys.prefix の直下、名前付き環境なら <base>/envs/<name> の 2 つ上に conda がある
    relative = Path("Scripts", "conda.exe") if os.name == "nt" else Path("bin", "conda")
    prefix = Path(sys.prefix)
    for root in (prefix, prefix.parent.parent):
        candidate = root / relative
        if candidate.is_file():
            return str(candidate)
    return None


conda_exe = find_conda()
if conda_exe is None:
    raise RuntimeError(
        "conda 実行ファイルが見つかりません。1-7 のつまずきポイントを参照してください。"
    )

command = [
    conda_exe,
    "install",
    "--prefix",
    sys.prefix,  # いま動いているカーネルの環境に入れる
    "--yes",
    "-c",
    "conda-forge",
    "pybullet",
    "numpy<2",  # panda-gym の必須制約。シェルを経由しないので引用符は不要
    "scipy",
]
print(">", " ".join(command), "\n")

#  解決とダウンロードに時間がかかるため、出力を 1 行ずつ流す
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace"
)
for line in process.stdout:
    print(line, end="")
returncode = process.wait()

if returncode != 0:
    raise RuntimeError(f"conda install に失敗しました（終了コード {returncode}）")
print("\nOK: conda-forge からの導入が完了しました。")


In [ ]:
# 1-3 (2/2) 残りを pip から導入する。バージョンは ../src/conda.yaml（Azure ML 側）と揃えてある
# stable-baselines3 が PyTorch を引き込むため、初回は数百 MB のダウンロードが発生する
%pip install "gymnasium==0.29.1" "panda-gym==3.0.7" "stable-baselines3==2.4.1" imageio imageio-ffmpeg pandas matplotlib


> [!IMPORTANT]
> **1-3 を実行したら、カーネルを再起動してください。**
> `numpy` のダウングレードなど、**既に読み込まれているモジュールが置き換わる**ことがあるためです。
>
> - VS Code: ノートブック上部の **［再起動］**
> - JupyterLab: **［Kernel］→［Restart Kernel］**
>
> 再起動しても、ここまでの導入結果は失われません。

### 1-4. 導入結果の検証

[../setup/verify_env.py](../setup/verify_env.py) を、**いま動いているカーネルの Python** で実行します。
セットアップ スクリプトと同じ検証を使うので、**ターミナルから実行した場合と結果が一致します。**

> [!NOTE]
> このスクリプトは **1-2 と 1-3 の両方**が入っていることを前提に検証します。
> **1-3 を飛ばした場合は「未導入」と表示されて失敗しますが、それは正常です。** その場合は 1-1 の診断結果で 1-2 の導入を確認してください。


In [ ]:
# ============================================================
#  1-4. 導入結果の検証
#  ../setup/verify_env.py を、いま動いているカーネルの Python で実行します。
#  ※ 1-2 と 1-3 の両方を実行した後に動きます（1-3 を飛ばした場合は失敗します）。
# ============================================================
import subprocess
import sys
from pathlib import Path

verify_script = Path("../setup/verify_env.py").resolve()
if not verify_script.is_file():
    raise FileNotFoundError(
        f"{verify_script} が見つかりません。"
        " このノートブックは 9.ReinforcementLearning/notebooks フォルダーを作業ディレクトリとして実行してください。"
    )

result = subprocess.run([sys.executable, str(verify_script)], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"終了コード: {result.returncode}（0 なら成功）")


### 1-5.【任意】RL 環境の中身を見てみる

**1-3 を実行した場合のみ動きます。** 1-4 の検証と重複しますが、こちらは**観測・行動の形をその場で確認する**ためのものです。


In [ ]:
# ============================================================
#  1-5. ローカル RL 環境の動作確認（1-3 を実行した場合のみ）
# ============================================================
import platform
import sys

import gymnasium as gym
import numpy as np
import panda_gym  # noqa: F401  # import すると Panda 系の環境が gymnasium に登録される

print("platform :", platform.system(), platform.machine())
print("python   :", sys.version.split()[0])
print("gymnasium:", gym.__version__)
print("numpy    :", np.__version__, "（panda-gym の制約により 2.x 未満である必要があります）")

# renderer="Tiny" は PyBullet の DIRECT 接続。ウィンドウを開かずに画像だけ取得する
env = gym.make("PandaReach-v3", render_mode="rgb_array", renderer="Tiny")
try:
    obs, info = env.reset(seed=0)
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    frame = np.asarray(env.render())

    print("observation shape:", obs["observation"].shape)
    print("action_space     :", env.action_space)
    print("render frame     :", frame.shape, frame.dtype)
finally:
    env.close()

print()
print("OK: ローカルで panda-gym が動作しています。")


### 1-6.【任意】GUI（3D シミュレーター）が開くかを確認する

`render_mode="human"` にすると、PyBullet の **OpenGL ウィンドウが別ウィンドウとして開きます。**
PyBullet 公式ドキュメントによれば、**Linux と Windows では GUI が別スレッドで動作します**（macOS のみ OS の制約で同一スレッド）。

> **出典（参考情報・OSS 公式ソース）**: PyBullet Quickstart Guide
> https://github.com/bulletphysics/bullet3/blob/master/docs/pybullet_quickstart_guide/PyBulletQuickstartGuide.md.html
> 「The GUI connection will create a new graphical user interface (GUI) with 3D OpenGL rendering ... **On Linux and Windows this GUI runs in a separate thread**, while on OSX it runs in the same thread due to operating system limitations.」

> [!WARNING]
> - **GUI ウィンドウはノートブックの中ではなく、別ウィンドウとして開きます。** 画面の裏に隠れていないか確認してください。
> - **`env.close()` を必ず実行してください。** 実行しないとウィンドウとバックグラウンドのスレッドが残ります。
> - **リモート デスクトップや仮想マシンでは OpenGL 3 が使えず、GUI の起動に失敗することがあります。** その場合の詳細は [04 章](../docs/04_RL環境を触って理解する.md) 4.6 を参照してください。


In [ ]:
# ============================================================
#  1-6. GUI の動作確認（手元の PC でのみ動きます）
#  ※ 別ウィンドウが開きます。数秒でロボットが動いて自動的に閉じます。
# ============================================================
import time

import gymnasium as gym
import panda_gym  # noqa: F401

env = gym.make("PandaPickAndPlace-v3", render_mode="human")
obs, info = env.reset(seed=0)

try:
    # ランダムな行動で少しだけ動かす（まだ学習していないので動きはでたらめです）
    for _ in range(60):
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        if terminated or truncated:
            obs, info = env.reset()

    # カメラ位置を変えられることの確認（詳細は docs/04 の 4.6.3）
    env.unwrapped.sim.place_visualizer(
        target_position=[0.0, 0.0, 0.0], distance=1.4, yaw=45, pitch=-30
    )
    time.sleep(2.0)
finally:
    env.close()  # ウィンドウとスレッドを必ず閉じる

print("OK: GUI が起動し、正常に終了しました。")


### 1-7. つまずきポイント（ローカル環境）

| 症状 | 原因 | 対処 |
|---|---|---|
| 1-3 の 1 つ目のセルが **`このカーネル（…）は conda 環境ではないため、conda で導入できません。`** で止まる | 現在のカーネルが conda 環境ではない | 1-1 の「conda 環境」表示を確認する。conda 環境を用意する手順は [../setup/setup.ps1](../setup/setup.ps1) / [../setup/setup.sh](../setup/setup.sh) |
| 1-3 の 1 つ目のセルが **`conda 実行ファイルが見つかりません。`** で止まる | conda 環境の Python でカーネルが起動しているが、`conda` 本体を特定できない | conda が導入されているフォルダーの `Scripts\conda.exe`（Windows）/ `bin/conda`（macOS・Linux）のパスを、環境変数 `CONDA_EXE` に設定してから VS Code を起動し直す |
| `error: Microsoft Visual C++ 14.0 or greater is required.` | `pybullet` を **PyPI から**入れようとしてソースビルドに入った（Windows / macOS 向けホイールが無いため） | **1-3 の 1 つ目のセル（conda）を先に実行する。** または [Microsoft C++ Build Tools](https://visualstudio.microsoft.com/visual-cpp-build-tools/) を導入する |
| conda の導入が `Cannot find a valid extracted directory cache` / `remove_all: The directory is not empty.` で失敗する | **パスが長すぎる。** Windows は既定でファイル パスの上限が 260 文字。`pybullet` パッケージは深い階層のファイルを含むため上限に達しやすい | **conda 環境とパッケージ キャッシュを短いパスに置く**（例: `C:\conda\envs`）。または「長いパスを有効にする」ポリシーを有効化する |
| 1-3 の 2 つ目のセル（pip）が非常に遅い | `stable-baselines3` が **PyTorch** を引き込むため（数百 MB のダウンロード） | 異常ではありません。完了まで待ってください |
| 1-3 の後も `import` が古いまま／`numpy` のバージョンが変わらない | **カーネルを再起動していない** | ノートブック上部の **［再起動］**（JupyterLab は ［Kernel］→［Restart Kernel］）を実行してから 1-4 へ |
| 1-4 が `NG: 次のパッケージが見つかりません` で終わる | 1-3 を実行していない（RL パッケージが未導入） | ローカルで RL を動かすなら 1-3 を実行する。**Azure ML にジョブを投げるだけなら 1-4 は不要**です |
| `numpy` が 2.x になっていて panda-gym が動かない | 後から入れたパッケージが numpy を 2.x に上げた | 1-3 の 1 つ目のセル（conda）を再実行する。`panda-gym` の `setup.py` が `numpy<2` を要求しています |
| GUI ウィンドウが開かない・真っ黒 | ウィンドウが背面に隠れている／リモート デスクトップや仮想マシンで OpenGL 3 が使えない | 画面を確認する。RDP・VM の場合は [04 章](../docs/04_RL環境を触って理解する.md) 4.6 を参照 |
| GUI を閉じてもプロセスが残る | `env.close()` を呼んでいない | 必ず `try` / `finally` で `env.close()` を呼ぶ |

> **出典（Microsoft Learn）**: [Maximum Path Length Limitation](https://learn.microsoft.com/windows/win32/fileio/maximum-file-path-limitation)
> 「the maximum length for a path is MAX_PATH, which is defined as 260 characters」／同ページに「Enable Long Paths in Windows 10, Version 1607, and Later」の有効化手順が記載されています。

> [!IMPORTANT]
> **ローカルと Azure ML では `pybullet` のバージョンが一致しません。**
>
> - **Azure ML 側**（[../src/conda.yaml](../src/conda.yaml)）は PyPI の `pybullet` を使います。
> - **ローカル側**は conda-forge の `pybullet` を使います。両者はバージョン番号の付け方が異なります。
>
> ローカルは **「動きを目で見て理解する」「コードのバグを潰す」ため**の環境です。
> **実験結果として記録・比較するのは、必ず Azure ML 上で実行したジョブにしてください**（[05 章](../docs/05_ベースライン実験.md)）。

> [!NOTE]
> **この構成は次の組み合わせで実際に動作を確認しています。**
>
> | 項目 | 値 |
> |---|---|
> | OS | Windows (x64) |
> | Python | 3.10（conda-forge） |
> | `pybullet` | conda-forge の `win-64` ビルド済みパッケージ |
> | `panda-gym` | 3.0.7（pip） |
> | `gymnasium` | 0.29.1（pip） |
> | `stable-baselines3` | 2.4.1（pip） |
> | `torch` | 2.13.0+cpu（pip が自動解決） |
> | `numpy` | 1.26.4 |
>
> **確認できた動作**
>
> - 環境の生成 / `reset()` / `step()` / `render()`（`Tiny`）
> - **GUI（`human`）ウィンドウの起動と終了**、`place_visualizer()` によるカメラ移動
> - `make_vec_env` による並列環境の生成
> - **SAC + `HerReplayBuffer` による学習**、モデルの保存・読み込み、`predict(deterministic=True)` による推論


## 2. ワークスペース情報の入力

**取得方法**: [Azure ML studio](https://ml.azure.com) の右上にあるワークスペース名をクリックすると、
サブスクリプション ID・リソース グループ・ワークスペース名が表示されます。

> 出典: [Create an Azure Machine Learning compute cluster - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-create-attach-compute-cluster?view=azureml-api-2)

In [ ]:
# ============================================================
#  ここを自分の環境に書き換えてください
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

#  本ハンズオンで作成するリソースの名前
COMPUTE_NAME = "cpu-cluster"
COMPUTE_SIZE = "Standard_DS3_v2"   # CPU 4 コア。クォータに合わせて調整してください
MAX_INSTANCES = 4                    # 並列実行できるジョブ数の上限
ENVIRONMENT_NAME = "rl-panda-gym-env"

#  コスト集計用タグ（docs/02 の 2.6 を参照）
TAGS = {
    "project": "rl-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
}

print("設定を読み込みました。")

## 3. Azure へのサインインとワークスペースへの接続

**サインインもここで行うので、ターミナルで `az login` を実行する必要はありません。**

`azure-identity` にはいくつかの認証方法があります。このノートブックでは `AUTH_MODE` で切り替えます。

| `AUTH_MODE` | 使うクラス | 動作 |
|---|---|---|
| `"auto"`（既定） | `DefaultAzureCredential` | 環境変数・マネージド ID・Visual Studio Code・Azure CLI・Azure PowerShell などを順に試し、**どれも使えなければブラウザーを開いて対話サインイン**します |
| `"browser"` | `InteractiveBrowserCredential` | 最初から既定のブラウザーを開いて対話サインインします |
| `"device"` | `DeviceCodeCredential` | **URL と確認コードを出力**します。別の端末のブラウザーでそのコードを入力してサインインします |

> [!IMPORTANT]
> **`"auto"` では `exclude_interactive_browser_credential=False` を明示的に指定しています。**
> Python の `DefaultAzureCredential` は、**既定では `InteractiveBrowserCredential` をチェーンから除外している**ためです。
> この指定が無いと、`az login` などを済ませていない環境ではトークンを取得できずに失敗します。
>
> **出典: Microsoft Learn** — [Credential chains in the Azure Identity library for Python](https://learn.microsoft.com/azure/developer/python/sdk/authentication/credential-chains#defaultazurecredential-overview)
> 「**`InteractiveBrowserCredential` is excluded by default** and therefore isn't shown in the preceding diagram. To include `InteractiveBrowserCredential`, **set the `exclude_interactive_browser_credential` keyword parameter to `False`** when you call the `DefaultAzureCredential` constructor.」

> [!NOTE]
> **ブラウザーが使えない環境（SSH 接続、GitHub Codespaces など）では `"device"` を使ってください。**
> 公式リファレンスは `DeviceCodeCredential` を「**Web ブラウザーの無い環境（SSH セッションなど）でユーザーを認証するのに主に有用**」と説明しています。
> また `prompt_callback` を指定しない場合、**認証手順（URL と確認コード）は標準出力に表示されます。**
>
> ⚠ `client_id` を指定しない場合、**Azure の開発用アプリケーションとして認証されます。** 公式リファレンスは「**運用環境のシナリオには推奨されない**」としており、本ハンズオンのような学習用途に限って使ってください。
>
> **出典: Microsoft Learn** — [DeviceCodeCredential Class](https://learn.microsoft.com/python/api/azure-identity/azure.identity.devicecodecredential?view=azure-python)

> [!WARNING]
> **対話サインインは、資格情報を入力し終えるまでセルの実行を止めます。**
> 公式ドキュメントは「**「対話型ブラウザー」認証は資格情報の入力を求める際にコード実行をブロックします。このアプローチはトレーニング ジョブなどの無人環境での認証には適しません**」と警告しています。
> 無人実行にはサービス プリンシパルを使ってください（**7.** の末尾を参照）。
>
> **出典: Microsoft Learn** — [Azure Machine Learning 用に MLflow を構成する](https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow-configure-tracking?view=azureml-api-2)

> **`az login` を既に済ませている場合、`"auto"` はその資格情報をそのまま使います**（チェーンに Azure CLI が含まれるため）。ブラウザーは開きません。
> **`az login` は必須ではありませんが、使っても構いません。**


In [ ]:
from azure.identity import (
    DefaultAzureCredential,
    DeviceCodeCredential,
    InteractiveBrowserCredential,
)

# ============================================================
#  ブラウザーが使えない環境では "device" に変えてください
# ============================================================
AUTH_MODE = "auto"   # "auto" | "browser" | "device"

if AUTH_MODE == "auto":
    #  Python の DefaultAzureCredential は既定で InteractiveBrowserCredential を除外するため、
    #  ここで明示的に有効化する。これで az login を済ませていなくてもサインインできる
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
elif AUTH_MODE == "browser":
    credential = InteractiveBrowserCredential()
elif AUTH_MODE == "device":
    #  別のテナントを使う場合は DeviceCodeCredential(tenant_id="<テナント ID>") と指定する
    credential = DeviceCodeCredential()
else:
    raise ValueError(f'AUTH_MODE は "auto" / "browser" / "device" のいずれかです（指定値: {AUTH_MODE}）')

print("認証方式:", type(credential).__name__)
print("※ 実際のサインインは次のセルで行われます。")
print("　 ブラウザーが開くか、確認コードが表示されることがあります。")


In [ ]:
from azure.ai.ml import MLClient

ml_client = MLClient(
    credential=credential,          # 上のセルで作った資格情報をそのまま使う
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

#  ここで初めて Azure へアクセスする。未サインインならこのタイミングで認証が始まる
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("Workspace      :", ws.name)
print("Location       :", ws.location)
print("Resource group :", ws.resource_group)


### ⚠ ここで失敗したら

**いずれもこのノートブックの中だけで対処できます。**

| 症状 | 対処 |
|---|---|
| トークンを取得できない（`ClientAuthenticationError` / `DefaultAzureCredential failed to retrieve a token`） | 前のセルの `AUTH_MODE` を **`"browser"`** に変えて、前のセルとこのセルを実行し直す |
| ブラウザーが開かない／リモート セッションで先へ進まない | `AUTH_MODE` を **`"device"`** に変えて実行し直す。表示された URL と確認コードを、手元の端末のブラウザーで入力する |
| 別のテナントのアカウントでサインインしてしまった | `AUTH_MODE` を `"device"` にしたうえで、`DeviceCodeCredential(tenant_id="<テナント ID>")` とテナントを指定する |
| サインインは通るがワークスペースが見つからない（`ResourceNotFound`） | **2.** のサブスクリプション ID / リソース グループ名 / ワークスペース名の綴りを確認する |
| `AuthorizationFailed` | 権限不足。[docs/02_Azure環境の準備.md](../docs/02_Azure環境の準備.md) の 2.1 に戻る |
| `NameError: name 'credential' is not defined` | 前の認証セルを実行していない。前のセルから順に実行する |


## 4. コンピューティング クラスターの作成

**最重要のパラメーター**

| パラメーター | 意味 |
|---|---|
| `min_instances=0` | **ジョブが無い間はノード数 0 → コンピューティングの課金が止まる** |
| `max_instances` | 並列実行できるジョブ数の上限（クォータを超えないこと） |
| `idle_time_before_scale_down=120` | アイドル 120 秒でノードを解放する |

> 出典: [Create an Azure Machine Learning compute cluster - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-create-attach-compute-cluster?view=azureml-api-2)

In [ ]:
from azure.ai.ml.entities import AmlCompute

try:
    cluster = ml_client.compute.get(COMPUTE_NAME)
    print(f"既存のクラスターを使います: {cluster.name} (size={cluster.size}, max={cluster.max_instances})")
except Exception:
    print("クラスターが無いので作成します（数分かかります）...")
    cluster = AmlCompute(
        name=COMPUTE_NAME,
        type="amlcompute",
        size=COMPUTE_SIZE,
        min_instances=0,
        max_instances=MAX_INSTANCES,
        idle_time_before_scale_down=120,
        tags=TAGS,
    )
    cluster = ml_client.begin_create_or_update(cluster).result()
    print(f"作成しました: {cluster.name}")

## 5. カスタム環境の作成

[../src/conda.yaml](../src/conda.yaml) をもとに、**ベース Docker イメージ ＋ conda 環境**の形で作成します。

> ⚠ **最重要**: Azure ML は **conda 定義から新しい環境を作り、その中でジョブを実行します。**
> **ベースイメージに入っている Python パッケージは使えません。** 必要なものはすべて `conda.yaml` に書いてください。
>
> 出典: [Manage Azure Machine Learning environments with the CLI and SDK (v2) - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-manage-environments-v2?view=azureml-api-2)

> ⚠ **panda-gym は `numpy<2` を要求します。** `conda.yaml` で固定済みです。

In [ ]:
from azure.ai.ml.entities import Environment

#  ベースイメージ。Microsoft Learn の環境作成サンプルで使用されているものです。
#  もしこのイメージが取得できない場合は、docs/03 のトラブルシューティング #6 を参照してください。
BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"

env = Environment(
    name=ENVIRONMENT_NAME,
    description="panda-gym + Stable-Baselines3 + MLflow (RL workshop)",
    image=BASE_IMAGE,
    conda_file="../src/conda.yaml",
    tags=TAGS,
)
env = ml_client.environments.create_or_update(env)

ENV_REF = f"{env.name}:{env.version}"
print("作成した環境:", ENV_REF)
print("※ 初回のイメージ構築には数分〜十数分かかります。studio の［環境］→［ビルド ログ］で進捗を確認できます。")

## 6. 疎通確認ジョブ

**ここまでの構築がすべて正しいかを 1 本のジョブで検証します。**

確認する項目:

1. panda-gym / Stable-Baselines3 が import できる
2. **`numpy<2` が守られている**
3. 環境を作って 1 エピソード動かせる
4. **MLflow にパラメーター・メトリック・成果物が記録される**

### 6-1. 疎通確認スクリプトを書き出す

In [ ]:
%%writefile ../src/smoke_test.py
"""Azure ML 疎通確認スクリプト。

Azure ML の Command Job として実行し、環境構築が正しいことを検証する。
ジョブとして実行される場合、MLflow の run は自動的に開始されるため
mlflow.start_run() は呼ばない。
  出典: https://learn.microsoft.com/azure/machine-learning/how-to-log-view-metrics?view=azureml-api-2
"""
import argparse
import json
import platform
import subprocess
import sys

import mlflow


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--env-id", type=str, default="PandaReach-v3")
    parser.add_argument("--seed", type=int, default=0)
    args = parser.parse_args()

    import numpy as np
    import gymnasium as gym
    import panda_gym  # noqa: F401  import すると環境 ID が Gymnasium に登録される
    import stable_baselines3 as sb3

    # --- 1) バージョンを記録する（再現性のために最重要） ---
    mlflow.log_param("env_id", args.env_id)
    mlflow.log_param("seed", args.seed)
    mlflow.log_param("python_version", platform.python_version())
    mlflow.log_param("numpy_version", np.__version__)
    mlflow.log_param("gymnasium_version", gym.__version__)
    mlflow.log_param("panda_gym_version", panda_gym.__version__)
    mlflow.log_param("sb3_version", sb3.__version__)

    # panda-gym は setup.py で numpy<2 を要求している
    assert np.__version__.startswith("1."), f"panda-gym は numpy<2 を要求します (現在: {np.__version__})"

    # --- 2) 環境を作って中身を確認する ---
    #  "Tiny" は PyBullet のソフトウェア レンダラーで、GPU も X サーバーも不要。
    #  ヘッドレスな Azure ML のコンピューティングではこれが必須。
    #  panda-gym v3 の既定値も render_mode="rgb_array" / renderer="Tiny" だが、
    #  バージョン差で renderer 引数を受け付けない場合に備えてフォールバックする。
    try:
        env = gym.make(args.env_id, render_mode="rgb_array", renderer="Tiny")
    except TypeError as exc:
        print(f"[WARN] renderer 引数を渡せませんでした ({exc})。render_mode のみで再試行します。")
        env = gym.make(args.env_id, render_mode="rgb_array")

    #  gym.make はラッパーを返すが、.spec から登録時の max_episode_steps を参照できる。
    #  万が一取得できない場合は panda-gym の登録値 50 を使う。
    max_steps = getattr(env.spec, "max_episode_steps", None) or 50

    print("observation_space :", env.observation_space)
    print("action_space      :", env.action_space)
    print("max_episode_steps :", max_steps)
    mlflow.log_param("max_episode_steps", max_steps)
    mlflow.log_param("action_dim", int(env.action_space.shape[0]))

    # --- 3) ランダム方策で 1 エピソード動かす ---
    obs, info = env.reset(seed=args.seed)
    print("observation keys  :", sorted(obs.keys()))

    total_reward = 0.0
    steps = 0
    for step in range(max_steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        steps += 1
        mlflow.log_metric("step_reward", float(reward), step=step)
        if terminated or truncated:
            break

    # --- 4) 描画がヘッドレスで動くか確認する（評価動画の前提） ---
    frame = env.render()
    render_ok = frame is not None
    if render_ok:
        print("rendered frame shape:", np.asarray(frame).shape)
    mlflow.log_metric("render_ok", float(render_ok))
    env.close()

    mlflow.log_metric("episode_reward", total_reward)
    mlflow.log_metric("episode_length", steps)
    mlflow.log_metric("is_success", float(bool(info.get("is_success", False))))
    print(f"episode_reward={total_reward}  steps={steps}  is_success={info.get('is_success')}")

    # --- 5) 成果物を記録する ---
    freeze = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True
    ).stdout
    with open("pip_freeze.txt", "w", encoding="utf-8") as fp:
        fp.write(freeze)
    mlflow.log_artifact("pip_freeze.txt")

    summary = {
        "env_id": args.env_id,
        "episode_reward": total_reward,
        "episode_length": steps,
        "render_ok": render_ok,
    }
    with open("smoke_summary.json", "w", encoding="utf-8") as fp:
        json.dump(summary, fp, ensure_ascii=False, indent=2)
    mlflow.log_artifact("smoke_summary.json")

    print("SMOKE TEST OK")


if __name__ == "__main__":
    main()

### 6-2. ジョブを投入する

In [ ]:
from azure.ai.ml import command

smoke_job = command(
    code="../src",                       # このフォルダー全体がスナップショットとして保存される
    command="python smoke_test.py --env-id PandaReach-v3 --seed 0",
    environment=ENV_REF,
    compute=COMPUTE_NAME,
    experiment_name="rl-setup-check",
    display_name="smoke_test_pandareach",
    tags=TAGS,
)

returned_job = ml_client.jobs.create_or_update(smoke_job)
print("ジョブ名 :", returned_job.name)
print("studio  :", returned_job.studio_url)

### 6-3. ジョブの完了を待つ

初回は**イメージ構築のため十数分かかることがあります**。
上のセルで表示された studio の URL を開くと、進捗とログをリアルタイムで確認できます。

In [ ]:
ml_client.jobs.stream(returned_job.name)

job = ml_client.jobs.get(returned_job.name)
print("ステータス:", job.status)

## 7. MLflow に記録された内容を確認する

**ここでもターミナルは不要です。**
`azureml-mlflow` プラグインは、**既定でブラウザーを開いて対話認証を行います。**
公式ドキュメントによれば、認証は次の順に試されます。

1. 環境（環境変数）
2. マネージド ID
3. **Azure CLI**（`az login` 済みならこれが使われます）
4. Azure PowerShell
5. **対話型ブラウザー**

> **出典: Microsoft Learn** — [Azure Machine Learning 用に MLflow を構成する](https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow-configure-tracking?view=azureml-api-2)
> 「By default, the Azure Machine Learning plugin for MLflow **performs interactive authentication by opening the default browser to prompt for credentials.**」

> [!NOTE]
> **MLflow プラグインは上記の順序で独自に認証を行います。**
> そのため、**3. でサインイン済みでも、このセクションでもう一度サインインを求められることがあります。**

> ⚠ **初心者がハマる仕様**
> `run.data.metrics` は、同じ名前のメトリックについて **最後の値しか返しません。**
> 全ステップの値（学習曲線）が欲しい場合は **`MlflowClient.get_metric_history()`** を使ってください。
>
> 出典: [Log metrics, parameters, and files with MLflow - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-log-view-metrics?view=azureml-api-2)


In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

#  Azure ML ワークスペースを MLflow のトラッキング先にする。
#  Azure ML のコンピューティング上で実行している場合は既に接続済みだが、
#  手元の PC から参照する場合は明示的な設定が必要。
#  出典: https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow-configure-tracking?view=azureml-api-2
try:
    tracking_uri = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
except AttributeError:
    #  フォールバック: 公式ドキュメントに記載されている形式で手動構築する
    print(
        "[WARN] SDK から追跡 URI を取得できませんでした。ドキュメント記載の形式で組み立てます。\n"
        "       Private Link を有効にしたワークスペースでは、この形式では接続できません。"
    )
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
print("tracking uri:", tracking_uri[:60], "...")

#  Azure ML のジョブ名を渡すと、対応する MLflow の run が取得できる
run = mlflow.get_run(returned_job.name)
run_id = run.info.run_id   # 以降はこの run_id を使う

print("=== パラメーター ===")
for k, v in sorted(run.data.params.items()):
    print(f"  {k:22s}: {v}")

print("\n=== メトリック（最終値のみ） ===")
for k, v in sorted(run.data.metrics.items()):
    print(f"  {k:22s}: {v}")

client = MlflowClient()
print("\n=== 成果物 ===")
for a in client.list_artifacts(run_id):
    print(" ", a.path)

print("\n=== step_reward の履歴（先頭5件） ===")
history = client.get_metric_history(run_id, "step_reward")
for m in history[:5]:
    print(f"  step={m.step}  value={m.value}")
print(f"  ... 合計 {len(history)} 点")


### 無人実行（サービス プリンシパル）を使う場合

対話サインインは、人が操作することを前提にしています。
**CI/CD やスケジュール実行など、人が居ない環境**では、サービス プリンシパルの情報を環境変数に設定してから実行します。

```python
import os

os.environ["AZURE_TENANT_ID"] = "<Azure-tenant-ID>"
os.environ["AZURE_CLIENT_ID"] = "<Azure-client-ID>"
os.environ["AZURE_CLIENT_SECRET"] = "<Azure-client-secret>"
```

この 3 つが設定されていると、**3. の `AUTH_MODE = "auto"`（`DefaultAzureCredential`）でも MLflow プラグインでも、チェーンの先頭にある「環境変数」の資格情報が使われます。** 対話サインインは発生しません。

> [!WARNING]
> **クライアント シークレットをノートブックやスクリプトに直接書かないでください。** Git に混入します。
> Azure Key Vault や、CI/CD パイプラインのシークレット機能から読み込んでください。

> **出典: Microsoft Learn** — [Azure Machine Learning 用に MLflow を構成する（認証の構成）](https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow-configure-tracking?view=azureml-api-2)
> サービス プリンシパルの作成手順は [Set up authentication for Azure Machine Learning resources and workflows](https://learn.microsoft.com/azure/machine-learning/how-to-setup-authentication?view=azureml-api-2#configure-a-service-principal) を参照してください。


## 8. ✅ チェックリスト（Azure 実験環境確認票）

**すべて満たしてから [docs/04_RL環境を触って理解する.md](../docs/04_RL環境を触って理解する.md) へ進んでください。**

### セットアップ（1.）

- [ ] 1-1 の診断で **`1-2 の実行 : 不要（導入済み）`** と表示された
- [ ] 【ローカルで RL を動かす場合のみ】1-1 の診断で **`1-3 の実行 : 不要（導入済み）`** と表示された
- [ ] 【同上】1-4 の検証が **`OK: ローカル環境の検証に成功しました。`**（終了コード 0）で終わった

### Azure（2. 〜 7.）

- [ ] **`az login` を実行せずに** `ws.name` が正しく表示された
- [ ] コンピューティング クラスターが **`min_instances=0`** で作成できた
- [ ] カスタム環境のビルドが成功した
- [ ] 疎通確認ジョブのステータスが **`Completed`** になった
- [ ] `numpy_version` が **`1.x`** であることを確認した（`2.x` なら [docs/03](../docs/03_AzureML環境構築.md) の TS #8 へ）
- [ ] `render_ok` が **`1.0`** であることを確認した（ヘッドレス描画が動く＝評価動画を作れる）
- [ ] 成果物に `pip_freeze.txt` と `smoke_summary.json` が表示された
- [ ] `step_reward` の履歴が複数点取得できた

### 通し

- [ ] **ここまで、ターミナルを一度も開かずに完了できた**

> **重要**: ここまで通れば、以降の演習は **「学習スクリプトを差し替えるだけ」** になります。

---

## ⚠ 後片付けのリマインド

**この日の作業を終えるときは、コンピューティング インスタンスを停止してください。**
クラスターは `min_instances=0` なので自動でノードが解放されますが、**インスタンスは手動停止（またはアイドル シャットダウン）が必要です。**

詳しい後片付け手順は [docs/09_評価・コスト・後片付け.md](../docs/09_評価・コスト・後片付け.md) にあります。
